In [25]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    brier_score_loss, log_loss
)


df = pd.read_csv('/Users/trevorpowell/mlproject/notebook/data/mlb_game_data.csv')
TARGET = "first_inning_run"

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET, "game_id", "date", "home_team", "away_team"])


X = pd.get_dummies(X, drop_first=False)

num_cols = [c for c in X.columns if np.issubdtype(X[c].dtype, np.number)]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols]   = scaler.transform(X_val[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

X_train_np = X_train.values.astype(np.float32)
X_val_np   = X_val.values.astype(np.float32)
X_test_np  = X_test.values.astype(np.float32)
y_train_np = y_train.values.astype(np.float32)
y_val_np   = y_val.values.astype(np.float32)
y_test_np  = y_test.values.astype(np.float32)

def eval_probs(y_true, p):
    return {
        "AUC": roc_auc_score(y_true, p),
        "AP": average_precision_score(y_true, p),
        "Brier": brier_score_loss(y_true, p),
        "LogLoss": log_loss(y_true, p),
        "Accuracy@0.5": accuracy_score(y_true, (p >= 0.5).astype(int)),
    }

In [26]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_np, y_train_np)

p_val_lr  = logreg.predict_proba(X_val_np)[:,1]
p_test_lr = logreg.predict_proba(X_test_np)[:,1]

print("Baseline (LogReg) – Val:", eval_probs(y_val_np, p_val_lr))
print("Baseline (LogReg) – Test:", eval_probs(y_test_np, p_test_lr))

Baseline (LogReg) – Val: {'AUC': 0.5204013377926421, 'AP': 0.5452647531906742, 'Brier': 0.25756395411935645, 'LogLoss': 0.7113283474674532, 'Accuracy@0.5': 0.5301724137931034}
Baseline (LogReg) – Test: {'AUC': 0.5426575505350774, 'AP': 0.5814915811115229, 'Brier': 0.25183441515141153, 'LogLoss': 0.6971911418434958, 'Accuracy@0.5': 0.5258620689655172}
